# Vision-Transformer CRP — Walkthrough

End-to-end tutorial for the four ViT concept-detector classes added in this fork, structured to mirror the original CRP repo's [`tutorials/attributions.ipynb`](../attributions.ipynb) and [`tutorials/feature_visualization.ipynb`](../feature_visualization.ipynb).

## What you'll see

1. **Setup** — imports, configuration knobs, paths.
2. **Dataset** — Imagenette-160 (10-class ImageNet subset, ~98 MB) with ImageNet-1k label mapping.
3. **Model + Canonizer + Composite** — load `vit_base_patch16_224`, build an `AttnLRPGammaComposite` (canonizer pre-bundled, no model-time patching), inspect what the canonizer does.
4. **Single-image conditional attribution** — pick a configurable target image, run a `HeadConcept`-conditioned backward pass, plot the heatmap.
5. **Build a `FeatureVisualization` index per concept granularity** — cached on disk; re-runs are no-ops.
6. **Top-concept identification + reference samples** — for each granularity (`HeadConcept`, `HeadDimConcept`, `KQVHeadConcept`, `KQVHeadDimConcept`), rank concepts under the target class, fetch the top-N samples that maximise each concept's relevance.
7. **Conditional heatmaps on the target image** — pixel-space attribution under the most-important concept of each granularity, side by side.

**Theory**: AttnLRP (Achtibat et al., ICML 2024; [arXiv 2402.05602](https://arxiv.org/abs/2402.05602)) on top of CRP (Achtibat et al., Nature MI 2023; [arXiv 2206.03208](https://arxiv.org/abs/2206.03208)).

**Concept-detector cheat sheet** (two orthogonal granularity dims):

| Class | Tap | Granularity | `attribute()` shape |
|---|---|---|---|
| `HeadConcept`        | `attn_out_tap` | per head (output tokens)             | `(B, num_heads)`              |
| `HeadDimConcept`     | `attn_out_tap` | per `(head, dim)` (output tokens)    | `(B, num_heads, head_dim)`    |
| `KQVHeadConcept`     | `qkv_tap`      | per `(part, head)` (K/Q/V projections) | `(B, 3, num_heads)`         |
| `KQVHeadDimConcept`  | `qkv_tap`      | per `(part, head, dim)` (K/Q/V projections) | `(B, 3, num_heads, head_dim)` |

Both taps are `nn.Identity` submodules installed by `AttentionTapsCanonizer`.

## 1. Setup

From the repo root:

```bash
uv sync --extra vit --extra dev --extra notebook
```

then launch this notebook with that env's kernel.

In [ ]:
from __future__ import annotations
import os
import urllib.request
import tarfile
from pathlib import Path

import numpy as np
import torch
import torchvision.transforms as T
import matplotlib.pyplot as plt
from PIL import Image

import timm
from timm.data import resolve_data_config
from torch.utils.data import Dataset

from crp.attention_concepts import (
    HeadConcept,
    HeadDimConcept,
    KQVHeadConcept,
    KQVHeadDimConcept,
    PARTS,
)
from crp.attribution import CondAttribution
from crp.transformer_patches import (
    AttnLRPEpsilonComposite,
    AttnLRPGammaComposite,
    AttentionTapsCanonizer,
    TimmViTCanonizer,
)
from crp.visualization import FeatureVisualization
from crp.image import plot_grid, vis_opaque_img

torch.set_grad_enabled(True)
print('torch', torch.__version__, '| timm', timm.__version__)

### 1.1 Configuration

All run-time knobs in one place. Override here for a GPU run or a bigger model.

* `MODEL_NAME` — `vit_base_patch16_224` (86 M) is the AttnLRP-paper default; `vit_small_patch16_224` (22 M) and `vit_tiny_patch16_224` (5.7 M) are CPU-friendly.
* `NUM_SAMPLES` — Imagenette images to index. 64–128 is plenty.
* `BLOCK_INDEX` — which ViT block to attribute. Mid-network blocks (5–8 in a 12-block ViT) carry the cleanest object-level concepts.
* `TARGET_INDEX` — index of the image we'll attribute. `None` → pick randomly from `RANDOM_SEED`.
* `GAMMA` — γ for the γ-LRP rule on linears (AttnLRP §3.2.1, default 0.25). Set `USE_GAMMA = False` to fall back to ε-LRP.

In [ ]:
MODEL_NAME = 'vit_base_patch16_224'   # 'vit_small_patch16_224' / 'vit_tiny_patch16_224' on CPU
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Dataset selection. ``imagenette`` (default) auto-downloads a 98 MB
# 10-class subset; ``imagenet_val`` requires a manually populated
# data/imagenet_val/ tree (gated on image-net.org / HuggingFace).
# Bumping NUM_SAMPLES gives nicer FV reference samples at the cost of
# longer indexing.
DATASET_NAME = 'imagenette'           # 'imagenette' | 'imagenet_val'
NUM_SAMPLES = 64
BLOCK_INDEX = 6
TOP_K = 4
TARGET_INDEX = None  # int → pick that index; None → random under RANDOM_SEED
RANDOM_SEED = 0

USE_GAMMA = True
GAMMA = 0.25
EPSILON = 1e-6

# Locate the repo root (so the notebook works whether launched from
# the repo root or from tutorials/vit_crp/). All artefacts live under
# <repo>/data/.
def _repo_root():
    p = Path.cwd().resolve()
    while p != p.parent:
        if (p / 'pyproject.toml').is_file():
            return p
        p = p.parent
    raise RuntimeError('repo root with pyproject.toml not found above CWD')
REPO_ROOT = _repo_root()
DATA_DIR = REPO_ROOT / 'data'
FV_ROOT = DATA_DIR / 'feature_visualization'
DATA_DIR.mkdir(parents=True, exist_ok=True)
FV_ROOT.mkdir(parents=True, exist_ok=True)

# Add experiments/ to sys.path so `import datasets` resolves to our
# loader (not the HuggingFace `datasets` package, were it installed).
import sys
sys.path.insert(0, str(REPO_ROOT / 'experiments'))

print(f'device  : {DEVICE}')
print(f'model   : {MODEL_NAME}')
print(f'dataset : {DATASET_NAME}')
print(f'samples : {NUM_SAMPLES}')
print(f'block   : {BLOCK_INDEX}')
print(f'rule    : {("γ-LRP, γ=" + str(GAMMA)) if USE_GAMMA else "ε-LRP"}')

## 2. Dataset

Dataset loading is delegated to [`experiments/datasets.py`](../../experiments/datasets.py). Two backends:

* **`imagenette`** — fast.ai's 10-class, 98 MB subset of real ImageNet images. Auto-downloaded to `data/imagenette2-160/` on first use. The default; good for development and laptop runs.
* **`imagenet_val`** — full 50 K-image ImageNet-1k validation split. **Not auto-downloaded** (the dataset is gated on image-net.org and HuggingFace). The loader expects `data/imagenet_val/<wnid>/<image>.JPEG` to be already populated — see the docstring in `datasets.py` for setup. Use this for final benchmark runs.

Both expose the same `CuratedDataset` interface: a `torch.utils.data.Dataset` yielding `(PIL.Image (RGB), int target_class)` pairs keyed by ImageNet-1k indices. The cell below downloads (if needed), filters to `NUM_SAMPLES` images, and returns the dataset object plus a wnid → 1k-index mapping for display.

In [ ]:
from datasets import load as load_dataset, IMAGENETTE_CLASS_NAMES

# For imagenette (10 classes), sample evenly: NUM_SAMPLES // 10 per class.
# For imagenet_val (1000 classes), use 1 image per class for a class-
# balanced subset.
if DATASET_NAME == 'imagenette':
    n_per_class = max(1, NUM_SAMPLES // 10)
    classes = None  # all 10
elif DATASET_NAME == 'imagenet_val':
    n_per_class = max(1, NUM_SAMPLES // 1000)
    classes = None  # all 1000
else:
    raise ValueError(DATASET_NAME)

dataset = load_dataset(
    DATASET_NAME,
    root=DATA_DIR,
    n_per_class=n_per_class,
    classes=classes,
    seed=RANDOM_SEED,
)
print(f'{dataset.name}: {len(dataset)} images, {dataset.num_classes} classes')
print(f'first 5 items:')
for path, cls in dataset.items[:5]:
    name = IMAGENETTE_CLASS_NAMES.get(cls, str(cls))
    print(f'  cls {cls:4d}  {name:25}  {path.name}')

## 3. Model + Canonizer + Composite

Standard zennit pipeline:

* **Canonizers** modify the model graph and forward methods so standard LRP rules can apply. We use `TimmViTCanonizer`, which composes `AttentionTapsCanonizer` (adds named `qkv_tap` and `attn_out_tap` `nn.Identity` submodules to every Attention) with `AttributeCanonizer`s that swap `forward` per-instance on `Attention`, `LayerNorm`, `GELU`, `Dropout` to embed the AttnLRP autograd functions in the forward pass.
* **Composite** maps module classes to LRP **Hooks**. We map `Linear` / `Conv2d` to a gradient×input ε or γ rule, and activations to `Pass` (the AttnLRP identity rule is already encoded in their forward via the canonizer).

All registration is **scoped to `composite.context()`**. No process-global state, no monkey-patching.

In [ ]:
model = timm.create_model(MODEL_NAME, pretrained=True).eval().to(DEVICE)

block = model.blocks[BLOCK_INDEX].attn
NUM_HEADS, HEAD_DIM = block.num_heads, block.head_dim
# Layer name is per-concept (each concept knows its tap):
QKV_LAYER = f'blocks.{BLOCK_INDEX}.attn.qkv_tap'
OUT_LAYER = f'blocks.{BLOCK_INDEX}.attn.attn_out_tap'

if USE_GAMMA:
    composite = AttnLRPGammaComposite(gamma=GAMMA, epsilon=EPSILON)
else:
    composite = AttnLRPEpsilonComposite(epsilon=EPSILON)

print(f'composite: {type(composite).__name__}')
print(f'qkv tap  : {QKV_LAYER}')
print(f'out tap  : {OUT_LAYER}')
print(f'num_heads: {NUM_HEADS}')
print(f'head_dim : {HEAD_DIM}')
print()
print('concept counts:')
print(f'  HeadConcept       -> {NUM_HEADS}')
print(f'  HeadDimConcept    -> {NUM_HEADS * HEAD_DIM}')
print(f'  KQVHeadConcept    -> {3 * NUM_HEADS}')
print(f'  KQVHeadDimConcept -> {3 * NUM_HEADS * HEAD_DIM}')

### 3.1 What the canonizer does (inspection cell)

Sanity-check that both taps only exist *inside* `composite.context()`. Before/after the `with` block, the model is exactly as `timm` constructed it.

In [ ]:
attn = model.blocks[BLOCK_INDEX].attn

print('before composite.context():')
print(f'  hasattr(attn, "qkv_tap")      = {hasattr(attn, "qkv_tap")}')
print(f'  hasattr(attn, "attn_out_tap") = {hasattr(attn, "attn_out_tap")}')

with composite.context(model) as modified:
    print()
    print('inside composite.context() (canonizer applied):')
    print(f'  hasattr(attn, "qkv_tap")      = {hasattr(attn, "qkv_tap")}')
    print(f'  hasattr(attn, "attn_out_tap") = {hasattr(attn, "attn_out_tap")}')

print()
print('after composite.context() exits (canonizer reverted):')
print(f'  hasattr(attn, "qkv_tap")      = {hasattr(attn, "qkv_tap")}')
print(f'  hasattr(attn, "attn_out_tap") = {hasattr(attn, "attn_out_tap")}')

### 3.2 Dataset wrapper + preprocess

`FeatureVisualization` expects `dataset[i]` to return `(unpreprocessed_tensor, int_target)`. Mean/std normalisation is applied by the `preprocess_fn` argument, so unpreprocessed tensors can be plotted directly as RGB images.

In [ ]:
cfg = resolve_data_config({}, model=model)
MEAN, STD, IMG_SIZE = cfg['mean'], cfg['std'], cfg['input_size'][1]

to_tensor = T.Compose([
    T.Resize(int(IMG_SIZE * 256 / 224)),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
])

MEAN_T = torch.tensor(MEAN).view(1, -1, 1, 1)
STD_T = torch.tensor(STD).view(1, -1, 1, 1)


def preprocess_fn(x):
    return (x - MEAN_T.to(x)) / STD_T.to(x)


def denormalize(x):
    if x.dim() == 3: x = x.unsqueeze(0)
    return x.detach().cpu().clamp(0, 1)[0].permute(1, 2, 0).numpy()


class ImagenetteDataset(Dataset):
    def __init__(self, root, num_samples):
        files, targets = [], []
        for wnid_dir in sorted((root / 'val').iterdir()):
            label = IMAGENETTE_TO_IMAGENET[wnid_dir.name]
            for f in sorted(wnid_dir.glob('*.JPEG')):
                files.append(f); targets.append(label)
        rng = np.random.default_rng(0)
        order = rng.permutation(len(files))[:num_samples]
        self.files = [files[i] for i in order]
        self.targets = [targets[i] for i in order]

    def __len__(self): return len(self.files)

    def __getitem__(self, i):
        img = Image.open(self.files[i]).convert('RGB')
        return to_tensor(img), int(self.targets[i])


dataset = ImagenetteDataset(EXTRACTED, NUM_SAMPLES)
print(f'dataset size: {len(dataset)}')

## 4. Single-image conditional attribution

Pick the target image (`TARGET_INDEX`, or random under `RANDOM_SEED`), run one `HeadConcept`-conditioned backward pass, and visualise the pixel-space heatmap. Same pattern as the original `attributions.ipynb` cell — only the `mask_map` and `composite` are different.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
if TARGET_INDEX is None:
    target_idx = int(rng.integers(0, len(dataset)))
else:
    target_idx = int(TARGET_INDEX)

target_data, target_class = dataset[target_idx]
target_pre = preprocess_fn(target_data.unsqueeze(0)).to(DEVICE)
target_pre.requires_grad_(True)

print(f'target sample idx : {target_idx}')
print(f'                    {dataset.files[target_idx].name}')
print(f'true class        : {target_class} ({CLASS_NAMES.get(target_class, "?")})')

with torch.no_grad():
    pred = model(target_pre)[0].softmax(dim=-1)
top5 = pred.topk(5)
print('top-5 model predictions:')
for prob, idx in zip(top5.values.tolist(), top5.indices.tolist()):
    name = CLASS_NAMES.get(idx, '')
    mark = ' <- target' if idx == target_class else ''
    print(f'  cls {idx:4d}  p={prob:.3f}  {name}{mark}')

In [ ]:
attribution = CondAttribution(model, device=torch.device(DEVICE))

head_concept = HeadConcept(model)

# Conditional attribution: HeadConcept head=0 under the target class.
conditions = [{OUT_LAYER: [0], 'y': [target_class]}]
result = attribution(
    target_pre, conditions, composite, mask_map=head_concept.mask,
)

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(denormalize(target_data))
axes[0].set_title(f'input  •  {CLASS_NAMES.get(target_class, target_class)}')
axes[0].axis('off')
hm = result.heatmap[0].detach().cpu().numpy()
vmax = np.abs(hm).max()
axes[1].imshow(denormalize(target_data), alpha=0.4)
axes[1].imshow(hm, cmap='bwr', alpha=0.7, vmin=-vmax, vmax=vmax)
axes[1].set_title(f'heatmap  •  HeadConcept head=0 @ block {BLOCK_INDEX}')
axes[1].axis('off')
plt.tight_layout(); plt.show()

### 4.1 Comparative single-image attribution across all four granularities

Run each of the four concept classes (`HeadConcept`, `HeadDimConcept`, `KQVHeadConcept`, `KQVHeadDimConcept`) on the same target image, pick the top-`TOP_K` concepts per granularity (ranked by absolute relevance under the target class), produce one conditional heatmap per concept, and render side by side. This is the previous standalone `demo.py`, folded into the notebook so all comparative results live in one place.

In [ ]:
from crp.attention_concepts import (
    HeadConcept, HeadDimConcept, KQVHeadConcept, KQVHeadDimConcept, PARTS,
)

CONCEPT_CLASSES = {
    'head':         HeadConcept,
    'head_dim':     HeadDimConcept,
    'kqv_head':     KQVHeadConcept,
    'kqv_head_dim': KQVHeadDimConcept,
}

def _enumerate_ids(name, num_heads, head_dim):
    if name == 'head':         return list(range(num_heads))
    if name == 'head_dim':     return [(h, d) for h in range(num_heads) for d in range(head_dim)]
    if name == 'kqv_head':     return [(p, h) for p in PARTS for h in range(num_heads)]
    if name == 'kqv_head_dim': return [(p, h, d) for p in PARTS for h in range(num_heads) for d in range(head_dim)]
    raise ValueError(name)

def _label(name, cid):
    if name == 'head':         return f'h{int(cid)}'
    if name == 'head_dim':     return f'h{int(cid[0])}/d{int(cid[1])}'
    if name == 'kqv_head':     return f'{cid[0]}/h{int(cid[1])}'
    if name == 'kqv_head_dim': return f'{cid[0]}/h{int(cid[1])}/d{int(cid[2])}'

NUM_HEADS, HEAD_DIM = model.blocks[BLOCK_INDEX].attn.num_heads, model.blocks[BLOCK_INDEX].attn.head_dim

rows = {}  # name → [(label, heatmap_np), ...]
for name, cls in CONCEPT_CLASSES.items():
    concept = cls(model)
    layer = f'blocks.{BLOCK_INDEX}.attn.{concept.tap_name}'
    # Per-concept relevance under the target class (no concept mask).
    target_pre.grad = None
    res = attribution(target_pre, [{'y': [target_class]}], composite,
                      mask_map=concept.mask, record_layer=[layer])
    scores = concept.attribute(res.relevances[layer], layer_name=layer, abs_norm=False)[0]
    all_ids = _enumerate_ids(name, NUM_HEADS, HEAD_DIM)
    k = min(TOP_K, len(all_ids))
    flat_top = torch.topk(scores.flatten().abs(), k=k).indices.tolist()
    top_ids = [all_ids[i] for i in flat_top]
    items = []
    for cid in top_ids:
        target_pre.grad = None
        r = attribution(target_pre, [{layer: [cid], 'y': [target_class]}], composite, mask_map=concept.mask)
        hm = r.heatmap[0]
        if hm.dim() == 3: hm = hm.sum(dim=0)
        items.append((_label(name, cid), hm.detach().cpu().numpy()))
    rows[name] = items

n_cols = TOP_K + 1
n_rows = len(rows)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(2 * n_cols, 2 * n_rows + 0.5))
img_np = denormalize(target_data)
for r_i, (name, items) in enumerate(rows.items()):
    axes[r_i, 0].imshow(img_np)
    axes[r_i, 0].set_title(name, fontsize=10)
    axes[r_i, 0].axis('off')
    for c_i, (label, hm) in enumerate(items, start=1):
        vmax = np.abs(hm).max()
        axes[r_i, c_i].imshow(img_np, alpha=0.4)
        axes[r_i, c_i].imshow(hm, cmap='bwr', alpha=0.6, vmin=-vmax, vmax=vmax)
        axes[r_i, c_i].set_title(label, fontsize=9)
        axes[r_i, c_i].axis('off')
    for c_i in range(len(items) + 1, n_cols):
        axes[r_i, c_i].axis('off')
fig.suptitle(f'block={BLOCK_INDEX}  •  target_class={target_class}', fontsize=11)
plt.tight_layout(); plt.show()

## 5. Build a FeatureVisualization index per concept granularity

For each of the four concept classes we build a separate FV index — same tap, different aggregation, different number of concepts. Each index ranks dataset samples by per-concept relevance under each sample's true class.

FV writes its results to `data/feature_visualization/<name>/` and the cell below skips `fv.run()` for any granularity that already has an index there. Delete that directory to force a rebuild.

In [ ]:
CONCEPT_DEFS = {
    'head':         HeadConcept,
    'head_dim':     HeadDimConcept,
    'kqv_head':     KQVHeadConcept,
    'kqv_head_dim': KQVHeadDimConcept,
}

concepts: dict = {}
fvs: dict = {}
layers: dict = {}
for name, cls in CONCEPT_DEFS.items():
    concept = cls(model)
    concepts[name] = concept
    layer_name = f'blocks.{BLOCK_INDEX}.attn.{concept.tap_name}'
    layers[name] = layer_name
    fvs[name] = FeatureVisualization(
        attribution,
        dataset,
        layer_map={layer_name: concept},
        preprocess_fn=preprocess_fn,
        path=str(FV_ROOT / name),
        device=torch.device(DEVICE),
    )

for name, layer in layers.items():
    print(f'  {name:13} -> {layer}')

In [ ]:
%%time
for name, fv in fvs.items():
    rel_dir = FV_ROOT / name / 'RelMax_sum_normed'
    has_index = rel_dir.is_dir() and any(rel_dir.glob('*.npy'))
    if has_index:
        print(f'[{name}] cached index found at {rel_dir} — skipping fv.run()')
        continue
    print(f'\n=== running FV index for {name!r} ===')
    fv.run(composite, 0, len(dataset), batch_size=8, checkpoint=10000)
print('\nall four indices ready.')

## 6. Top-concept identification + reference samples

For the chosen target image:

1. Run a backward pass per granularity, recording relevance at the concept's tap (`attn_out_tap` for output-side concepts, `qkv_tap` for K/Q/V-side concepts), masked by `concept.mask` under the target class.
2. Aggregate via `concept.attribute()` to get one scalar per concept id.
3. Take the top-K concepts by absolute relevance.
4. From the FV index, fetch the top-N **reference samples** that maximise each concept's relevance over the dataset.
5. Render with `crp.image.plot_grid`.

In [ ]:
def per_concept_scores(concept, layer_name, data, target_class):
    conditions = [{'y': [target_class]}]
    result = attribution(
        data, conditions, composite,
        mask_map=concept.mask, record_layer=[layer_name],
    )
    rel = result.relevances[layer_name]
    return concept.attribute(rel, layer_name=layer_name, abs_norm=False)[0]


def top_k_flat(scores, k):
    flat = scores.flatten()
    k = min(k, flat.numel())
    return torch.topk(flat.abs(), k=k).indices.tolist()


def label_for(name, flat_id):
    if name == 'head':
        return f'h{flat_id}'
    if name == 'head_dim':
        h, d = divmod(flat_id, HEAD_DIM)
        return f'h{h}/d{d}'
    if name == 'kqv_head':
        p, h = divmod(flat_id, NUM_HEADS)
        return f'{PARTS[p]}/h{h}'
    if name == 'kqv_head_dim':
        p, rem = divmod(flat_id, NUM_HEADS * HEAD_DIM)
        h, d = divmod(rem, HEAD_DIM)
        return f'{PARTS[p]}/h{h}/d{d}'
    raise ValueError(name)


top_ids: dict = {}
for name, concept in concepts.items():
    target_pre.grad = None
    scores = per_concept_scores(concept, layers[name], target_pre, target_class)
    ids = top_k_flat(scores, TOP_K)
    top_ids[name] = ids
    pretty = ', '.join(label_for(name, i) for i in ids)
    print(f'{name:>9s}: top-{TOP_K}  {pretty}')

### 6.1 Reference samples + per-sample conditional heatmaps

`get_max_reference(..., composite=composite, plot_fn=vis_opaque_img)` returns, for each requested concept id, the top-N samples that maximise its relevance over the dataset, with a conditional heatmap rendered as an opacity mask on each — so non-relevant pixels fade out and the user sees *which patch of the reference image the concept fires on*. This is the canonical RelMax interpretation pattern from the CRP paper.

`vis_opaque_img` is shipped in `crp.image`. `plot_grid` renders the resulting `{concept_id: PIL.Image}` dict directly.

In [ ]:
REF_RANGE = (0, 4)  # top-1..top-4 reference sample per concept

for name, ids in top_ids.items():
    print(f'\n── {name} ──')
    fv = fvs[name]
    ref_c = fv.get_max_reference(
        ids, layers[name], mode='relevance', r_range=REF_RANGE,
        composite=composite, plot_fn=vis_opaque_img,
    )
    pretty = {label_for(name, k): v for k, v in ref_c.items()}
    plot_grid(pretty, figsize=(7, 1.6 * len(ids)), padding=False)
    plt.show()

## 7. Conditional heatmaps on the target image

For each granularity's most-important concept (rank 0 from §6), compute a pixel-space attribution conditioned on **just that concept** under the target class. Side-by-side comparison shows how concept granularity trades off localisation vs. interpretability:

- `head` covers the whole head's contribution — broadest support;
- `kqv` covers a whole projection (Q, K, or V) across all heads;
- `kqv_head` is the intersection — narrower;
- `head_dim` is a single feature dimension — sharpest, sometimes noisy.

In [ ]:
def conditional_heatmap(concept, layer_name, concept_id, data, target_class):
    conditions = [{layer_name: [concept_id], 'y': [target_class]}]
    result = attribution(data, conditions, composite, mask_map=concept.mask)
    hm = result.heatmap[0]
    if hm.dim() == 3:
        hm = hm.sum(dim=0)
    return hm.detach().cpu().numpy()


img_np = denormalize(target_data)
fig, axes = plt.subplots(1, 5, figsize=(2.4 * 5, 2.6))

axes[0].imshow(img_np)
axes[0].set_title(f'input\n{CLASS_NAMES.get(target_class, target_class)}')
axes[0].axis('off')

for i, (name, ids) in enumerate(top_ids.items()):
    ax = axes[i + 1]
    cid = ids[0]  # most important
    target_pre.grad = None
    hm = conditional_heatmap(concepts[name], layers[name], cid, target_pre, target_class)
    vmax = np.abs(hm).max()
    ax.imshow(img_np, alpha=0.4)
    ax.imshow(hm, cmap='bwr', alpha=0.7, vmin=-vmax, vmax=vmax)
    ax.set_title(f'{name}\n{label_for(name, cid)}')
    ax.axis('off')

fig.suptitle(
    f'conditional heatmaps  •  block={BLOCK_INDEX}  •  '
    f'composite={type(composite).__name__}',
    fontsize=11,
)
plt.tight_layout(); plt.show()

## 8. What's next

* Try a different `BLOCK_INDEX` — early blocks (0–3) tend to encode low-level features (edges, colour); late blocks (9–11) encode object-/class-level semantics.
* Re-pick the target image (`TARGET_INDEX = …` or change `RANDOM_SEED`) — concepts will shift.
* Switch composites (`USE_GAMMA = False` for ε-LRP) and rebuild the FV indices (delete `data/feature_visualization/`) to compare γ vs. ε qualitatively.
* Use `compute_stats` / `get_stats_reference` (see [`tutorials/feature_visualization.ipynb`](../feature_visualization.ipynb)) to find the dataset class for which each concept is most representative.
* Read [`experiments/metrics.py`](../../experiments/metrics.py) for the deletion / insertion AUC faithfulness benchmark across the four granularities and the random-concept baseline.

**Outstanding work**: see [`FUTURE_STATE.md`](../../FUTURE_STATE.md) — stability metric, localisation metric, multi-block comparison figure, broader baselines (gradient-only, Grad-CAM, occlusion).